# Fracture Fixation Nails – What FDA Reports Reveal
Case Study 5 · Dr. Muhammad Ali, DPT

**Question:** What problems are reported for intramedullary fracture nails (the rods placed inside long bones such as the femur and tibia), what happens to patients, and how many problems relate to healing, loading and rehabilitation (e.g. nail breakage, non-union, loosening)?

**Data:** FDA MAUDE adverse-event reports via the openFDA API, product code **HSB** ("rod, fixation, intramedullary and accessories").

**Gate logic:** every step ends with automatic checks. If a number does not match the FDA's own count, the notebook stops with `GATE FAILED`.

## Step 1 – Set up the project and plan the download
The openFDA API can return at most 25,000 reports per query. The FDA total for HSB is above that, so the reports per year are requested first; the download in Step 2 then runs year by year.

**Gates:** the FDA answers · reports per year add up to the FDA total · no single year exceeds 25,000.

In [1]:
# 1.1 Set up folders, define the gate check, get the FDA total for HSB and the reports per year
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, gzip, requests
import pandas as pd

BASE  = '/content/drive/MyDrive/fracture-fixation-fda'
RAW, CLEAN, FIG = f'{BASE}/data/raw', f'{BASE}/data/clean', f'{BASE}/figures'
for folder in [RAW, CLEAN, FIG]:
    os.makedirs(folder, exist_ok=True)

API     = 'https://api.fda.gov/device/event.json'
QUERY   = 'device.device_report_product_code:HSB'
HEADERS = {'User-Agent': 'Mozilla/5.0 (research notebook)'}

def gate(condition, message):
    """Stop the notebook if a check fails; print a pass line if it succeeds."""
    if not condition:
        raise AssertionError('GATE FAILED: ' + message)
    print('GATE PASSED:', message)

r = requests.get(API, params={'search': QUERY, 'limit': 1}, headers=HEADERS, timeout=60)
gate(r.status_code == 200, 'FDA API answered (status 200)')
meta = r.json()['meta']
FDA_TOTAL = meta['results']['total']
print('FDA database last updated:', meta['last_updated'])
print('Reports with product code HSB:', f'{FDA_TOTAL:,}')

# Reports per day received -> add up per year
days = requests.get(API, params={'search': QUERY, 'count': 'date_received'}, headers=HEADERS, timeout=60).json()['results']
per_year = (pd.DataFrame(days).assign(year=lambda d: d['time'].str[:4].astype(int))
              .groupby('year')['count'].sum())
gate(per_year.sum() == FDA_TOTAL, f'reports per year add up to the FDA total ({per_year.sum():,})')
gate(per_year.max() <= 25000, f'largest single year ({per_year.max():,}) is within the 25,000 limit')
print('\nReports per year:\n', per_year.to_string())

Mounted at /content/drive
GATE PASSED: FDA API answered (status 200)
FDA database last updated: 2026-09-22
Reports with product code HSB: 37,835
GATE PASSED: reports per year add up to the FDA total (37,835)
GATE PASSED: largest single year (4,111) is within the 25,000 limit

Reports per year:
 year
1992       6
1993      15
1994      19
1995      24
1996      53
1997      92
1998      74
1999      51
2000      43
2001      38
2002      57
2003      76
2004      61
2005      78
2006     114
2007     162
2008     184
2009     334
2010     454
2011     662
2012    1079
2013    1773
2014    1474
2015    1671
2016    1920
2017    1447
2018    3526
2019    2517
2020    3299
2021    4111
2022    2733
2023    2218
2024    2869
2025    2933
2026    1668


## Step 2 – Download every report, year by year
Because the total is above the API's 25,000-report limit, the reports are downloaded one year at a time (by date received), 100 per request, and combined. The combined data is saved **untouched** as `data/raw/hsb_reports_raw.jsonl.gz`.

**Gates:**
1. Every single year: downloaded reports = the FDA's count for that year (35 years checked)
2. Total downloaded = FDA total
3. Every report key is unique and present
4. The saved file, read back from Drive, contains every report

In [2]:
# 2.1 Download every HSB report year by year (100 per request), check each year, save the raw data
PAGE = 100
records, year_check = [], {}

for year, expected_n in per_year.items():
    q = f'{QUERY} AND date_received:[{year}0101 TO {year}1231]'
    got = []
    for skip in range(0, expected_n, PAGE):
        for attempt in range(5):                                   # retry if the FDA server is busy
            r = requests.get(API, params={'search': q, 'limit': PAGE, 'skip': skip}, headers=HEADERS, timeout=120)
            if r.status_code == 200:
                break
            time.sleep(5 * (attempt + 1))
        if r.status_code != 200:
            print('FDA error message:', r.text[:500])
            raise RuntimeError(f'Download failed: year {year}, skip={skip}, status {r.status_code}')
        got.extend(r.json()['results'])
        time.sleep(0.4)
    year_check[year] = (len(got), expected_n)
    records.extend(got)
    print(f'{year}: {len(got):,} of {expected_n:,}   (total so far {len(records):,})', end='\r')
print()

mismatched = {y: v for y, v in year_check.items() if v[0] != v[1]}
gate(not mismatched, f'all {len(year_check)} years match the FDA count for that year' + (f' – mismatches: {mismatched}' if mismatched else ''))
gate(len(records) == FDA_TOTAL, f'downloaded reports ({len(records):,}) = FDA total ({FDA_TOTAL:,})')

keys = [rec.get('mdr_report_key') for rec in records]
gate(all(keys), 'every report has a report key')
gate(len(set(keys)) == len(keys), 'every report key is unique (no duplicates)')

raw_path = f'{RAW}/hsb_reports_raw.jsonl.gz'
with gzip.open(raw_path, 'wt', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec) + '\n')
with gzip.open(raw_path, 'rt', encoding='utf-8') as f:
    saved = sum(1 for _ in f)
gate(saved == FDA_TOTAL, f'saved raw file contains all {saved:,} reports')
print(f'Raw file size: {os.path.getsize(raw_path) / 1e6:.1f} MB')

2026: 1,668 of 1,668   (total so far 37,835)
GATE PASSED: all 35 years match the FDA count for that year
GATE PASSED: downloaded reports (37,835) = FDA total (37,835)
GATE PASSED: every report has a report key
GATE PASSED: every report key is unique (no duplicates)
GATE PASSED: saved raw file contains all 37,835 reports
Raw file size: 20.6 MB


## Step 3 – Turn the reports into a table, check it against the FDA, and save it
Every report becomes **one row**: date received, event type (death / injury / malfunction), manufacturer, brand name, coded device and patient problems, and the event description. The table is saved as `data/clean/nail_reports.csv`.

**Gates:**
1. One row per report (= FDA total), unique report keys, every report has a valid date
2. **Independent check:** the counts of deaths, injuries, malfunctions, other and blank event types equal the FDA's own count, requested separately from the API
3. The saved file, read back from Drive, has the same rows and the same event-type counts

*Event types left blank by the reporter are kept as "(blank)", because the FDA counts them as a category too.*

In [3]:
# 3.1 Flatten the reports into one row per report, check against the FDA's own counts, and save
def join_values(values, sep='; '):
    """Join a list into text, skipping empty values (some FDA lists contain None)."""
    return sep.join(sorted({str(v).strip() for v in (values or []) if v not in (None, '')}))

def report_text(rec, kind):
    return ' '.join((t.get('text') or '') for t in (rec.get('mdr_text') or [])
                    if t.get('text_type_code') == kind).strip()

rows = []
for rec in records:
    devices = rec.get('device') or []
    rows.append({
        'report_key':       rec.get('mdr_report_key'),
        'date_received':    rec.get('date_received'),
        'event_type':       rec.get('event_type') or '',
        'manufacturer':     join_values([d.get('manufacturer_d_name') for d in devices], ' | '),
        'brand':            join_values([d.get('brand_name') for d in devices], ' | '),
        'product_problems': join_values(rec.get('product_problems')),
        'patient_problems': join_values([p for pt in (rec.get('patient') or []) for p in (pt.get('patient_problems') or [])]),
        'event_text':       report_text(rec, 'Description of Event or Problem'),
        'mfr_text':         report_text(rec, 'Additional Manufacturer Narrative'),
    })
df = pd.DataFrame(rows)
df['date_received'] = pd.to_datetime(df['date_received'], format='%Y%m%d', errors='coerce')

gate(len(df) == FDA_TOTAL, f'table has one row per report ({len(df):,})')
gate(df['report_key'].is_unique, 'report keys are unique in the table')
gate(df['date_received'].notna().all(), 'every report has a valid date')

fda_counts = requests.get(API, params={'search': QUERY, 'count': 'event_type.exact'},
                          headers=HEADERS, timeout=60).json()['results']
fda_counts = {c['term']: c['count'] for c in fda_counts}
our_counts = df['event_type'].value_counts().to_dict()
gate(our_counts == fda_counts, 'death / injury / malfunction / other / blank counts match the FDA exactly')
gate(sum(our_counts.values()) == FDA_TOTAL, f'event-type groups add up to the full total ({FDA_TOTAL:,})')

out_path = f'{CLEAN}/nail_reports.csv'
df.to_csv(out_path, index=False)
check = pd.read_csv(out_path, dtype={'report_key': str}, keep_default_na=False)
gate(len(check) == FDA_TOTAL, 'saved file has every report')
gate(check['event_type'].value_counts().to_dict() == our_counts, 'event-type counts identical after saving')

print('\nEvent type     Our table      FDA')
for k in sorted(fda_counts, key=fda_counts.get, reverse=True):
    print(f'{(k or "(blank)"):<14} {our_counts.get(k, 0):>9,} {fda_counts[k]:>8,}')
print('\nDate range:', df['date_received'].min().date(), 'to', df['date_received'].max().date())
print('Top 8 manufacturers:\n', df['manufacturer'].value_counts().head(8).to_string())

GATE PASSED: table has one row per report (37,835)
GATE PASSED: report keys are unique in the table
GATE PASSED: every report has a valid date
GATE PASSED: death / injury / malfunction / other / blank counts match the FDA exactly
GATE PASSED: event-type groups add up to the full total (37,835)
GATE PASSED: saved file has every report
GATE PASSED: event-type counts identical after saving

Event type     Our table      FDA
Injury            25,103   25,103
Malfunction       12,477   12,477
Death                115      115
Other                108      108
(blank)               32       32

Date range: 1992-01-08 to 2026-08-31
Top 8 manufacturers:
 manufacturer
SYNTHES GMBH                             6109
STRYKER TRAUMA KIEL                      5142
SIGN FRACTURE CARE INTERNATIONAL         2717
WRIGHTS LANE SYNTHES USA PRODUCTS LLC    2604
STRYKER GMBH                             2547
SMITH & NEPHEW, INC.                     2491
OBERDORF SYNTHES PRODUKTIONS GMBH        1916
ZIMMER BIO